# 04 — High-frequency accelerometer observations

**Goal:** reason about high-frequency data and compare raw ingestion with aggregation.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
DATA = ROOT / 'data'

from sta_client import BASE_URL, get_json, post_entity, post_observation
BASE_URL


In [ ]:
import pandas as pd
acc = pd.read_csv(DATA / 'accelerometer_sample.csv')
acc.head()

## Exercise 4.1 — Compute simple features over 1-second windows

In [ ]:
acc['time'] = pd.to_datetime(acc['phenomenonTime'])
acc = acc.set_index('time')
features = acc[['ax_m_s2','ay_m_s2','az_m_s2']].resample('1s').agg(['mean','std','min','max'])
features.head()

## Exercise 4.2 — Create a compact observation payload
Instead of posting every 50 Hz raw sample, post a JSON result containing aggregated features.

In [ ]:
ACC_DATASTREAM_ID = None  # TODO

first_window = features.iloc[0].dropna()
result = {f'{axis}_{stat}': float(value) for (axis, stat), value in first_window.items()}
payload = {
    'phenomenonTime': features.index[0].isoformat().replace('+00:00','Z'),
    'result': result,
    'Datastream': {'@iot.id': ACC_DATASTREAM_ID},
    'parameters': {'window': '1s', 'raw_source': 'accelerometer_sample.csv'}
}
payload

In [ ]:
# TODO: send the aggregated observation
# post_entity('Observations', payload)